In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb # Regression model
from sklearn.model_selection import train_test_split # To split training date for testing
import lightgbm as lgb

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/signals093/sample_submission.csv
/kaggle/input/signals093/train.csv
/kaggle/input/signals093/test.csv


In [2]:
print("Loading data...")
df_train = pd.read_csv("/kaggle/input/signals093/train.csv")
df_test = pd.read_csv("/kaggle/input/signals093/test.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Loading data...
Train shape: (76500, 22)
Test shape: (25500, 21)


In [3]:
df_train['regime'] = df_train['regime'].astype('category')
df_test['regime'] = df_test['regime'].astype('category')

features = [f'signal_{i:02d}' for i in range(1,19)] + ['regime']
target = 'target_value'
weight_col = 'sample_weight'

x = df_train[features]
y = df_train[target]
w = df_train[weight_col]

x_test = df_test[features]

In [4]:
def weighted_rmse(y_true, y_pred, w):
    numerator = np.sum(w * (y_true - y_pred)**2)
    weighted_mean = np.average(y_true, weights=w)
    denominator = np.sum(w * (y_true - weighted_mean)**2)
    
    return 1 - (numerator/denominator)

In [5]:
print("Starting seed blending...")

n_folds = 5
test_preds_sum = np.zeros(len(df_test))
val_scores = []

for i in range(n_folds):
    seed = 42 + i
    print(f"\n--- Training model {i+1}/{n_folds} (Seed {seed}) ---")

    x_train, x_val, y_train, y_val, w_train, w_val = train_test_split(x, y, w, test_size=0.2, random_state=seed)

    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=5000,
        learning_rate=0.01,
        max_depth=9,
        subsample=0.7,
        colsample_bytree=0.7,
        enable_categorical=True,
        n_jobs=-1,
        random_state=seed,
        early_stopping_rounds=100
    )
    
    print("Training model...")
    model.fit(
        x_train, y_train,
        sample_weight=w_train,
        eval_set=[(x_val, y_val)],
        verbose=False
    )

    fold_preds = model.predict(x_val)
    fold_score = weighted_rmse(y_val, fold_preds, w_val)
    val_scores.append(fold_score)
    print(f"Model {i+1} Validation score: {fold_score: .5f}")
    
    test_preds_sum += model.predict(x_test)

Starting seed blending...

--- Training model 1/5 (Seed 42) ---
Training model...
Model 1 Validation score:  0.86583

--- Training model 2/5 (Seed 43) ---
Training model...
Model 2 Validation score:  0.86632

--- Training model 3/5 (Seed 44) ---
Training model...
Model 3 Validation score:  0.86257

--- Training model 4/5 (Seed 45) ---
Training model...
Model 4 Validation score:  0.86901

--- Training model 5/5 (Seed 46) ---
Training model...
Model 5 Validation score:  0.86493


In [6]:
print("Starting LightGBM Training")

n_folds = 5
lgb_test_preds_sum = np.zeros(len(df_test))
lgb_val_scores = []

for i in range(n_folds):
    seed = 42 + i
    print(f"\n--- Training LightGBM {i+1}/{n_folds} (Seed {seed}) ---")

    x_train, x_val, y_train, y_val, w_train, w_val = train_test_split(x, y, w, test_size=0.2, random_state=seed)

    model_lgb = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=5000,
        learning_rate=0.01,
        num_leaves=31,
        max_depth=8,
        subsample=0.7,
        colsample_bytree=0.7,
        n_jobs=-1,
        random_state=seed,
        importance_type='gain'
    )

    cat_cols = ['regime'] 
    
    model_lgb.fit(
        x_train, y_train,
        sample_weight=w_train,
        eval_set=[(x_val, y_val)],
        eval_metric='rmse',
        categorical_feature=cat_cols,
        callbacks=[
            lgb.early_stopping(stopping_rounds=100),
            lgb.log_evaluation(period=0)
        ]
    )

    fold_preds = model_lgb.predict(x_val)
    fold_score = weighted_rmse(y_val, fold_preds, w_val)
    lgb_val_scores.append(fold_score)
    print(f"LGBM {i+1} Validation Score: {fold_score:.5f}")

    lgb_test_preds_sum += model_lgb.predict(x_test)

Starting LightGBM Training

--- Training LightGBM 1/5 (Seed 42) ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007416 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4135
[LightGBM] [Info] Number of data points in the train set: 61200, number of used features: 19
[LightGBM] [Info] Start training from score -1.303374
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[5000]	valid_0's rmse: 1.32558
LGBM 1 Validation Score: 0.87161

--- Training LightGBM 2/5 (Seed 43) ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006815 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4135
[LightGBM] [Info] Number of 

In [7]:
final_test_preds = test_preds_sum / n_folds
final_lgb_preds = lgb_test_preds_sum / n_folds

average_score = np.mean(val_scores)
lgb_average_score = np.mean(lgb_val_scores)

print(f"\nAverage LightGBM Score: {lgb_average_score: .5f}")
print(f"\nAverage Validation Score: {average_score: .5f}")

ensemble_preds = (0.5 * final_test_preds) + (0.5 * final_lgb_preds)

submission = pd.DataFrame({
    'sample_id': df_test['sample_id'],
    'target_value': ensemble_preds
})

submission.to_csv('submission.csv', index=False)
print("submission.csv saved!")


Average LightGBM Score:  0.86950

Average Validation Score:  0.86573
submission.csv saved!
